# Gmail Credential Setup for Event Register

This notebook creates or refreshes your Gmail OAuth token for this project.

It uses the same scopes as `email_client.py` and writes the token to `email_token.json`.

## Before you run
1. In Google Cloud Console, enable **Gmail API** and **People API**.
2. Create an **OAuth client ID** for Desktop App.
3. Download the client secret file and place it in this folder as `credentials.json`.

In [1]:
# Install required Google auth libraries (safe to re-run)
%pip install --upgrade google-api-python-client google-auth-httplib2 google-auth-oauthlib

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.0.1 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
from pathlib import Path
import json

from google.auth.transport.requests import Request
from google.oauth2.credentials import Credentials
from google_auth_oauthlib.flow import InstalledAppFlow
from googleapiclient.discovery import build
from google.auth.exceptions import RefreshError

# Match email_client.py exactly
SCOPES = [
    "https://www.googleapis.com/auth/gmail.readonly",
    "https://www.googleapis.com/auth/gmail.modify",
    "https://www.googleapis.com/auth/contacts.readonly",
]

CREDENTIALS_FILE = Path("credentials.json")
TOKEN_FILE = Path("email_token.json")

print(f"Working directory: {Path.cwd()}")
print(f"Credentials file: {CREDENTIALS_FILE.resolve()}")
print(f"Token file: {TOKEN_FILE.resolve()}")
print(f"Scopes: {json.dumps(SCOPES, indent=2)}")

Working directory: c:\Users\haas1\programming\event_register
Credentials file: C:\Users\haas1\programming\event_register\credentials.json
Token file: C:\Users\haas1\programming\event_register\email_token.json
Scopes: [
  "https://www.googleapis.com/auth/gmail.readonly",
  "https://www.googleapis.com/auth/gmail.modify",
  "https://www.googleapis.com/auth/contacts.readonly"
]


In [3]:
if not CREDENTIALS_FILE.exists():
    raise FileNotFoundError(
        "credentials.json not found. Download OAuth Desktop credentials from Google Cloud and place the file here."
    )

print("credentials.json found.")

credentials.json found.


In [4]:
creds = None

# Load existing token if present
if TOKEN_FILE.exists():
    try:
        creds = Credentials.from_authorized_user_file(str(TOKEN_FILE), SCOPES)
        print("Loaded existing token from email_token.json")
    except RefreshError:
        print("Existing token is invalid/revoked. Removing it.")
        TOKEN_FILE.unlink(missing_ok=True)
        creds = None

# Refresh or start OAuth flow
if not creds or not creds.valid:
    if creds and creds.expired and creds.refresh_token:
        print("Refreshing expired token...")
        try:
            creds.refresh(Request())
        except RefreshError:
            print("Refresh failed. Starting new OAuth flow...")
            creds = None

    if not creds:
        print("Starting OAuth browser flow...")
        flow = InstalledAppFlow.from_client_secrets_file(str(CREDENTIALS_FILE), SCOPES)
        creds = flow.run_local_server(port=0)

TOKEN_FILE.write_text(creds.to_json(), encoding="utf-8")
print("Saved token to email_token.json")

Loaded existing token from email_token.json
Refreshing expired token...
Saved token to email_token.json


In [5]:
# Verify Gmail access by reading your profile
gmail_service = build("gmail", "v1", credentials=creds)
profile = gmail_service.users().getProfile(userId="me").execute()
print("Authenticated Gmail address:", profile.get("emailAddress"))
print("Messages total:", profile.get("messagesTotal"))

Authenticated Gmail address: robin.pickleball.scheduler@gmail.com
Messages total: 719


In [6]:
# Optional: quick People API check (contacts scope)
people_service = build("people", "v1", credentials=creds)
connections = people_service.people().connections().list(
    resourceName="people/me",
    personFields="emailAddresses",
    pageSize=5,
).execute()

count = len(connections.get("connections", []))
print(f"People API reachable. Sample contacts returned: {count}")

People API reachable. Sample contacts returned: 2


## Done
If the profile cell shows your Gmail address, your credentials are ready.

Your project should now be able to authenticate via `email_token.json`.